# Gene Ontology Enrichment Analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load Libraries

In [ ]:
!pip install -q "pandas==2.2.2" "numpy<2.1" gseapy scanpy anndata scikit-learn

In [ ]:
import os
import math
import json
import tempfile
import warnings
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

import gseapy as gp

## Configuration


In [ ]:
AGGREGATION_TOP_N = 10
FDR_CUTOFF = 0.05
TOP_N_TERMS_FOR_PLOTS = 10

GENE_SET_LIBRARIES = [
    "GO_Biological_Process_2025",
    "GO_Molecular_Function_2025",
    "GO_Cellular_Component_2025",
    "KEGG_2021_Human",
    "Reactome_2022",
]

BACKGROUND_FILE = "/content/drive/.../background_genes.txt"
BACKGROUND_GENES = None


OUTDIR = "/content/drive/.../go_enrichment_results"
LIBRARY_CACHE_DIR = os.path.join(OUTDIR, "_library_cache")

KNOWN_TECHNICAL_ARTIFACT_GENES = {
    "MALAT1": "Ubiquitous nuclear lncRNA; well-documented dissociation/technical-stress "
              "marker in scRNA-seq (van den Brink et al. 2017 and follow-ups); negligible "
              "informative GO_BP annotation.",
    "RPPH1": "RNase P RNA component (ncRNA); housekeeping, negligible GO_BP annotation.",
}

KNOWN_GENE_ALIASES = {
    "CCN5": "WISP2",
}

NEGATIVE_CONTROL_PATHS = {
    "VQFS_PQK_Random_SVM-RBF",
    "VQFS_PQK_Random_RandomForest",
    "VQFS_PQK_Random_LogisticRegression",
}

PREFERRED_LIBRARY_ORDER = [
    "GO_Biological_Process_2025",
    "KEGG_2021_Human",
    "GO_Cellular_Component_2025",
    "GO_Molecular_Function_2025",
    "Reactome_2022",
]


## Per-Fold Gene Lists

In [ ]:
FOLD_GENE_SETS = {
"Classical_PCA_SVM-RBF": {
"D1": ["COL1A1","COL3A1","COL1A2","S100A9","KRT6A","S100A8","S100A2","S100A7","SPARC","POSTN"],
"D2": ["S100A9","KRT6A","S100A8","S100A7","S100A2","COL1A1","COL1A2","COL3A1","KRT6B","KRT17"],
"D3": ["S100A9","COL1A1","COL3A1","KRT6A","COL1A2","S100A8","S100A7","KRT17","S100A2","KRT6B"],
"D4": ["S100A9","COL1A1","COL3A1","COL1A2","KRT6A","S100A8","S100A7","S100A2","KRT17","KRT6B"]
},
"Classical_PCA_RandomForest": {
"D1": ["COL1A1","COL3A1","COL1A2","S100A9","KRT6A","S100A8","S100A2","S100A7","SPARC","POSTN"],
"D2": ["S100A9","KRT6A","S100A8","S100A7","S100A2","COL1A1","COL1A2","COL3A1","KRT6B","KRT17"],
"D3": ["S100A9","COL1A1","COL3A1","KRT6A","COL1A2","S100A8","S100A7","KRT17","S100A2","KRT6B"],
"D4": ["S100A9","COL1A1","COL3A1","COL1A2","KRT6A","S100A8","S100A7","S100A2","KRT17","KRT6B"]
},
"Classical_PCA_LogisticRegression": {
"D1": ["COL1A1","COL3A1","COL1A2","S100A9","KRT6A","S100A8","S100A2","S100A7","SPARC","POSTN"],
"D2": ["S100A9","KRT6A","S100A8","S100A7","S100A2","COL1A1","COL1A2","COL3A1","KRT6B","KRT17"],
"D3": ["S100A9","COL1A1","COL3A1","KRT6A","COL1A2","S100A8","S100A7","KRT17","S100A2","KRT6B"],
"D4": ["S100A9","COL1A1","COL3A1","COL1A2","KRT6A","S100A8","S100A7","S100A2","KRT17","KRT6B"]
},
"Classical_SparsePCA_SVM-RBF": {
"D1": ["YPEL2","CDK11A","UHRF1BP1L","MKLN1","EHBP1","HOXC8","PCNX2","XPC","INPP5D","USP53"],
"D2": ["SMG6","ZNF407","ZNF493","ENPP1","ARHGAP24","NHLRC3","TTC23","UGGT2","CHID1","SAMD4A"],
"D3": ["CDK11B","TTC14","DAPK1","INPP5D","KIF20B","NTN1","NAV2","DOCK4","ZNF609","TSHZ2"],
"D4": ["ZNF641","DIAPH2","MSL2","FBXO33","AR","GARS1-DT","UGGT2","CSGALNACT1","POLR2J3","SIPA1L2"]
},
"Classical_SparsePCA_RandomForest": {
"D1": ["YPEL2","CDK11A","UHRF1BP1L","MKLN1","EHBP1","HOXC8","PCNX2","XPC","INPP5D","USP53"],
"D2": ["SMG6","ZNF407","ZNF493","ENPP1","ARHGAP24","NHLRC3","TTC23","UGGT2","CHID1","SAMD4A"],
"D3": ["CDK11B","TTC14","DAPK1","INPP5D","KIF20B","NTN1","NAV2","DOCK4","ZNF609","TSHZ2"],
"D4": ["ZNF641","DIAPH2","MSL2","FBXO33","AR","GARS1-DT","UGGT2","CSGALNACT1","POLR2J3","SIPA1L2"]
},
"Classical_SparsePCA_LogisticRegression": {
"D1": ["YPEL2","CDK11A","UHRF1BP1L","MKLN1","EHBP1","HOXC8","PCNX2","XPC","INPP5D","USP53"],
"D2": ["SMG6","ZNF407","ZNF493","ENPP1","ARHGAP24","NHLRC3","TTC23","UGGT2","CHID1","SAMD4A"],
"D3": ["CDK11B","TTC14","DAPK1","INPP5D","KIF20B","NTN1","NAV2","DOCK4","ZNF609","TSHZ2"],
"D4": ["ZNF641","DIAPH2","MSL2","FBXO33","AR","GARS1-DT","UGGT2","CSGALNACT1","POLR2J3","SIPA1L2"]
},
"Classical_ICA_SVM-RBF": {
"D1": ["HBB","TPSB2","HBA1","CCL21","TPSAB1","TFF3","CPA3","HBA2","MMRN1","CTSG"],
"D2": ["HBA1","HBB","HBA2","TPSB2","CPA3","TPSAB1","TFF3","CTSG","MS4A2","CCL21"],
"D3": ["HBB","HBA1","TPSB2","TFF3","HBA2","TPSAB1","CPA3","FADS2","CTSG","CCL21"],
"D4": ["HBB","HBA1","HBA2","TPSB2","CPA3","TPSAB1","TFF3","CCL21","RERGL","CTSG"]
},
"Classical_ICA_RandomForest": {
"D1": ["HBB","TPSB2","HBA1","CCL21","TPSAB1","TFF3","CPA3","HBA2","MMRN1","CTSG"],
"D2": ["HBA1","HBB","HBA2","TPSB2","CPA3","TPSAB1","TFF3","CTSG","MS4A2","CCL21"],
"D3": ["HBB","HBA1","TPSB2","TFF3","HBA2","TPSAB1","CPA3","FADS2","CTSG","CCL21"],
"D4": ["HBB","HBA1","HBA2","TPSB2","CPA3","TPSAB1","TFF3","CCL21","RERGL","CTSG"]
},
"Classical_ICA_LogisticRegression": {
"D1": ["HBB","TPSB2","HBA1","CCL21","TPSAB1","TFF3","CPA3","HBA2","MMRN1","CTSG"],
"D2": ["HBA1","HBB","HBA2","TPSB2","CPA3","TPSAB1","TFF3","CTSG","MS4A2","CCL21"],
"D3": ["HBB","HBA1","TPSB2","TFF3","HBA2","TPSAB1","CPA3","FADS2","CTSG","CCL21"],
"D4": ["HBB","HBA1","HBA2","TPSB2","CPA3","TPSAB1","TFF3","CCL21","RERGL","CTSG"]
},
"VQFS_PQK_MI_SVM-RBF": {
"D1": ["GSN","KRT10","KRT1","LORICRIN","CCN5","TNC","ADH1B","FBLN1","CFD","APOD"],
"D2": ["MT2A","S100A9","S100A7","CFD","GSN","LTBP4","FABP5","S100A8","FBLN1","TNC"],
"D3": ["CFD","MALAT1","TNXB","LTBP4","SERPING1","DCN","MT2A","S100A9","CCN5","APOD"],
"D4": ["GSN","S100A9","S100A8","LTBP4","TNC","KRT6A","CFD","MALAT1","ADIRF","FBLN1"]
},
"VQFS_PQK_MI_RandomForest": {
"D1": ["GSN","KRT10","KRT1","LORICRIN","CCN5","TNC","ADH1B","FBLN1","CFD","APOD"],
"D2": ["MT2A","S100A9","S100A7","CFD","GSN","LTBP4","FABP5","S100A8","FBLN1","TNC"],
"D3": ["CFD","MALAT1","TNXB","LTBP4","SERPING1","DCN","MT2A","S100A9","CCN5","APOD"],
"D4": ["GSN","S100A9","S100A8","LTBP4","TNC","KRT6A","CFD","MALAT1","ADIRF","FBLN1"]
},
"VQFS_PQK_MI_LogisticRegression": {
"D1": ["GSN","KRT10","KRT1","LORICRIN","CCN5","TNC","ADH1B","FBLN1","CFD","APOD"],
"D2": ["MT2A","S100A9","S100A7","CFD","GSN","LTBP4","FABP5","S100A8","FBLN1","TNC"],
"D3": ["CFD","MALAT1","TNXB","LTBP4","SERPING1","DCN","MT2A","S100A9","CCN5","APOD"],
"D4": ["GSN","S100A9","S100A8","LTBP4","TNC","KRT6A","CFD","MALAT1","ADIRF","FBLN1"]
},
"VQFS_PQK_mRMR_SVM-RBF": {
"D1": ["RPPH1","CTC-246B18.10","MT2A","GAB2","SIN3A","GSN","CFD","SUPT7L","OCRL","ATP6V0E2"],
"D2": ["NTHL1","STXBP1","SLC35B3","ZBED1","GSN","NSMCE1","CFD","USP33","RBAK","IFRD1"],
"D3": ["CFD","MAD1L1","USP33","TTBK2","EIF2AK3","ZNF264","FBLN1","TJAP1","TSC2","GPATCH1"],
"D4": ["GTPBP1","CFD","SART1","GK5","MALAT1","MED25","FBLN1","MYG1","YPEL2","THSD4"]
},
"VQFS_PQK_mRMR_RandomForest": {
"D1": ["SLC8A1","ZDHHC24","CFD","KCTD6","FAM207A","NFIL3","PEX26","ALPK1","AUH","TBC1D12"],
"D2": ["GSN","ATF1","ACO1","C8orf82","SLC35B3","MT2A","FBLN1","TOP3B","C8orf33","CFD"],
"D3": ["RTL8A","CFD","BCLAF3","FBLN1","PINK1","TMEM231","BBS10","PELP1","MARF1","NSMF"],
"D4": ["CFD","INPP5B","MALAT1","ZNF747","TRIM35","RAD51D","RAPH1","IPO13","PON2","FBLN1"]
},
"VQFS_PQK_mRMR_LogisticRegression": {
"D1": ["QTRT2","FBXO33","INO80","MTMR9","FAM207A","CFD","MTAP","CCDC112","RBM7","MTHFR"],
"D2": ["DNMBP","MGRN1","GSN","ITGB3","CDK11A","GPAT4","RTL8A","ZNF3","CFD","CCDC14"],
"D3": ["DHRS3","CFD","EIF5B","FBLN1","ADAMTS1","WDR24","CACFD1","ZNF777","CD68","CCDC71L"],
"D4": ["FLCN","CFD","MTFR1L","TP53INP2","NDUFV3","RCOR3","NES","MALAT1","APPBP2","KLHL25"]
},
"VQFS_PQK_Variance_SVM-RBF": {
"D1": ["S100A9","S100A2","S100A7","POSTN","S100A8","CCDC80","KRT6A","SPARC","LUM","KRT1"],
"D2": ["S100A9","S100A7","S100A8","KRT14","S100A2","DCN","CCDC80","COL3A1","KRT6A","KRT17"],
"D3": ["S100A9","POSTN","LUM","KRT14","S100A8","COL1A1","DCN","KRT17","S100A2","KRT6A"],
"D4": ["S100A9","KRT14","POSTN","DCN","KRT6A","KRT17","S100A8","COL3A1","COL1A1","CCDC80"]
},
"VQFS_PQK_Variance_RandomForest": {
"D1": ["S100A9","S100A2","S100A7","POSTN","S100A8","CCDC80","KRT6A","SPARC","LUM","KRT1"],
"D2": ["S100A9","S100A7","S100A8","KRT14","S100A2","DCN","CCDC80","COL3A1","KRT6A","KRT17"],
"D3": ["S100A9","POSTN","LUM","KRT14","S100A8","COL1A1","DCN","KRT17","S100A2","KRT6A"],
"D4": ["S100A9","KRT14","POSTN","DCN","KRT6A","KRT17","S100A8","COL3A1","COL1A1","CCDC80"]
},
"VQFS_PQK_Variance_LogisticRegression": {
"D1": ["S100A9","S100A2","S100A7","POSTN","S100A8","CCDC80","KRT6A","SPARC","LUM","KRT1"],
"D2": ["S100A9","S100A7","S100A8","KRT14","S100A2","DCN","CCDC80","COL3A1","KRT6A","KRT17"],
"D3": ["S100A9","POSTN","LUM","KRT14","S100A8","COL1A1","DCN","KRT17","S100A2","KRT6A"],
"D4": ["S100A9","KRT14","POSTN","DCN","KRT6A","KRT17","S100A8","COL3A1","COL1A1","CCDC80"]
},
"VQFS_PQK_Random_SVM-RBF": {
"D1": ["GREM2","RECK","PRMT7","SCARA5","NAV3","ARRDC2","TP53INP2","SVIL","APOL1","SNIP1"],
"D2": ["S100A9","RBBP9","DTYMK","ZBED1","PPM1L","MROH6","CYP27A1","LRCH3","SLC48A1","MGRN1"],
"D3": ["CENATAC","BSCL2","IFITM10","ZDHHC8","MRGPRF","TAF3","TCEA2","RBM25","WDR7","SLC41A3"],
"D4": ["MGMT","KLC3","GPRC5C","HPCAL1","DCT","SHTN1","EML4","PPP1R12B","CPQ","C2CD3"]
},
"VQFS_PQK_Random_RandomForest": {
"D1": ["GREM2","RECK","PRMT7","SCARA5","NAV3","ARRDC2","TP53INP2","SVIL","APOL1","SNIP1"],
"D2": ["S100A9","RBBP9","DTYMK","ZBED1","PPM1L","MROH6","CYP27A1","LRCH3","SLC48A1","MGRN1"],
"D3": ["CENATAC","BSCL2","IFITM10","ZDHHC8","MRGPRF","TAF3","TCEA2","RBM25","WDR7","SLC41A3"],
"D4": ["MGMT","KLC3","GPRC5C","HPCAL1","DCT","SHTN1","EML4","PPP1R12B","CPQ","C2CD3"]
},
"VQFS_PQK_Random_LogisticRegression": {
"D1": ["GREM2","RECK","PRMT7","SCARA5","NAV3","ARRDC2","TP53INP2","SVIL","APOL1","SNIP1"],
"D2": ["S100A9","RBBP9","DTYMK","ZBED1","PPM1L","MROH6","CYP27A1","LRCH3","SLC48A1","MGRN1"],
"D3": ["CENATAC","BSCL2","IFITM10","ZDHHC8","MRGPRF","TAF3","TCEA2","RBM25","WDR7","SLC41A3"],
"D4": ["MGMT","KLC3","GPRC5C","HPCAL1","DCT","SHTN1","EML4","PPP1R12B","CPQ","C2CD3"]
},
"MI_Classical_Baseline_SVM-RBF": {
"D1": ["DCN","CCN5","KRT1","KRT10","ADH1B","GSN","FBLN1","S100A9","CFD","LORICRIN"],
"D2": ["DCN","CCN5","S100A7","GSN","FBLN1","IGFBP6","S100A9","S100A8","CFD","TNXB"],
"D3": ["MALAT1","LTBP4","DCN","CCN5","GSN","FBLN1","S100A9","S100A8","CFD","TNXB"],
"D4": ["KRT6A","MALAT1","DCN","CCN5","GSN","FBLN1","IGFBP6","S100A9","S100A8","CFD"]
},
"MI_Classical_Baseline_RandomForest": {
"D1": ["DCN","CCN5","KRT1","KRT10","ADH1B","GSN","FBLN1","S100A9","CFD","LORICRIN"],
"D2": ["DCN","CCN5","S100A7","GSN","FBLN1","IGFBP6","S100A9","S100A8","CFD","TNXB"],
"D3": ["MALAT1","LTBP4","DCN","CCN5","GSN","FBLN1","S100A9","S100A8","CFD","TNXB"],
"D4": ["KRT6A","MALAT1","DCN","CCN5","GSN","FBLN1","IGFBP6","S100A9","S100A8","CFD"]
},
"MI_Classical_Baseline_LogisticRegression": {
"D1": ["DCN","CCN5","KRT1","KRT10","ADH1B","GSN","FBLN1","S100A9","CFD","LORICRIN"],
"D2": ["DCN","CCN5","S100A7","GSN","FBLN1","IGFBP6","S100A9","S100A8","CFD","TNXB"],
"D3": ["MALAT1","LTBP4","DCN","CCN5","GSN","FBLN1","S100A9","S100A8","CFD","TNXB"],
"D4": ["KRT6A","MALAT1","DCN","CCN5","GSN","FBLN1","IGFBP6","S100A9","S100A8","CFD"]
},
"QuantumKernel_CPSS_mRMR_SVM": {
"D1": ["PPP1R15B","MYSM1","ABCB6","CYB5D2","TPRKB","MAP3K7","WDR12","TRA2B","GSN","ITGAE"],
"D2": ["GOLPH3","LMNA","AHNAK","CYB5R3","SORBS3","GSN","LGALS3","LAMP1","MYDGF","IFNGR2"],
"D3": ["NOP53","PDIA3","SELENOW","LGALS3","PCBP2","SURF4","CFD","AHNAK","STAT2","SDF4"],
"D4": ["TMED7","QSOX1","PAPOLA","MAP4","LRP10","HM13","CYB5R3","LAMP1","LGALS3","CFD"]
}
}

## Fold aggregation (consensus gene selection across LODO folds)

In [ ]:
def aggregate_genes_across_folds(fold_gene_lists, top_n=AGGREGATION_TOP_N):
    n_folds = len(fold_gene_lists)
    gene_ranks = defaultdict(list)
    gene_folds = defaultdict(list)

    for fold_name, genes in fold_gene_lists.items():
        for rank, gene in enumerate(genes):
            gene_ranks[gene.upper()].append(rank)
            gene_folds[gene.upper()].append(fold_name)

    rows = []
    for gene, ranks in gene_ranks.items():
        rows.append({
            "gene": gene,
            "n_folds_selected": len(ranks),
            "n_folds_total": n_folds,
            "selection_frequency": len(ranks) / n_folds,
            "mean_rank_when_selected": sum(ranks) / len(ranks),
            "folds": ", ".join(gene_folds[gene]),
        })

    detail_df = pd.DataFrame(rows).sort_values(
        by=["n_folds_selected", "mean_rank_when_selected", "gene"],
        ascending=[False, True, True],  # explicit alphabetical tiebreak
    ).reset_index(drop=True)

    top_genes = detail_df.head(top_n)["gene"].tolist()
    return top_genes, detail_df


def build_gene_sets_from_folds(fold_gene_sets, outdir, top_n=AGGREGATION_TOP_N):
    agg_dir = os.path.join(outdir, "_gene_aggregation")
    os.makedirs(agg_dir, exist_ok=True)
    gene_sets = {}

    print("=== Aggregating per-fold gene lists into one consensus list per path ===")
    for path_name, fold_lists in fold_gene_sets.items():
        top_genes, detail_df = aggregate_genes_across_folds(fold_lists, top_n=top_n)
        detail_df.to_csv(os.path.join(agg_dir, f"{path_name}_fold_aggregation.csv"), index=False)
        gene_sets[path_name] = top_genes
        n_unanimous = (detail_df["n_folds_selected"] == len(fold_lists)).sum()
        print(f"[{path_name}] {len(fold_lists)} folds -> {len(top_genes)} consensus genes "
              f"({n_unanimous} selected in ALL folds): {top_genes}")
    return gene_sets


def deduplicate_gene_sets(gene_sets):
    signature_to_canonical = {}
    alias_map = {}
    unique_gene_sets = {}

    for path_name, genes in gene_sets.items():
        signature = tuple(sorted(genes))
        if signature not in signature_to_canonical:
            signature_to_canonical[signature] = path_name
            unique_gene_sets[path_name] = genes
        alias_map[path_name] = signature_to_canonical[signature]

    dup_groups = defaultdict(list)
    for path_name, canonical in alias_map.items():
        dup_groups[canonical].append(path_name)

    print("\n=== De-duplication of identical gene sets across paths ===")
    for canonical, members in dup_groups.items():
        if len(members) > 1:
            print(f"[dedup] {members} share IDENTICAL gene sets -> "
                  f"computing enrichment once as '{canonical}', reusing for the rest.")
    return unique_gene_sets, alias_map


## Gene-list curation: alias expansion + artifact-gene flagging

In [ ]:
def curate_query_genes(genes):
    log = {"aliases_expanded": {}, "artifacts_removed": {}}
    curated = list(dict.fromkeys(g.upper() for g in genes))  # de-dup, preserve order

    expanded = []
    for g in curated:
        expanded.append(g)
        if g in KNOWN_GENE_ALIASES:
            alias = KNOWN_GENE_ALIASES[g]
            expanded.append(alias)
            log["aliases_expanded"][g] = alias
    curated = list(dict.fromkeys(expanded))

    kept = []
    for g in curated:
        if g in KNOWN_TECHNICAL_ARTIFACT_GENES:
            log["artifacts_removed"][g] = KNOWN_TECHNICAL_ARTIFACT_GENES[g]
        else:
            kept.append(g)

    return kept, log


## Background Gene Universe

In [ ]:
def load_background():
    if BACKGROUND_GENES is not None:
        return sorted({g.upper() for g in BACKGROUND_GENES})
    if os.path.exists(BACKGROUND_FILE):
        with open(BACKGROUND_FILE) as f:
            genes = [line.strip().upper() for line in f if line.strip()]
        return sorted(set(genes))
    raise FileNotFoundError(
        f"No background file found at '{BACKGROUND_FILE}' and BACKGROUND_GENES is not set. "
        f"Run export_shared_background_full_panel(adata_paths) first. Refusing to silently "
        f"fall back to a whole-genome default background — that would invalidate the "
        f"hypergeometric test's null model."
    )


def export_shared_background_full_panel(
    adata_paths, out_txt=BACKGROUND_FILE,
):
    import scanpy as sc

    if isinstance(adata_paths, str):
        adata_paths = [adata_paths]

    panels = []
    for p in adata_paths:
        adata = sc.read_h5ad(p)
        panel = set(g.upper() for g in adata.var_names)
        panels.append(panel)
        print(f"[background] {p}: {len(panel)} genes in panel")

    reference = panels[0]
    for p, panel in zip(adata_paths[1:], panels[1:]):
        if panel != reference:
            only_in_ref = reference - panel
            only_in_other = panel - reference
            raise ValueError(
                f"Gene panels differ between adata files — this would silently "
                f"reintroduce a background mismatch across paths.\n"
                f"  In '{adata_paths[0]}' but not '{p}': {len(only_in_ref)} genes "
                f"(e.g. {sorted(only_in_ref)[:10]})\n"
                f"  In '{p}' but not '{adata_paths[0]}': {len(only_in_other)} genes "
                f"(e.g. {sorted(only_in_other)[:10]})\n"
                f"Confirm all three notebooks are reading copies of the SAME "
                f"bio_dataset.h5ad (e.g. compare file hashes) before proceeding."
            )

    genes = sorted(reference)
    with open(out_txt, "w") as f:
        f.write("\n".join(genes))
    print(f"[background] Shared full-panel background: {len(genes)} genes -> {out_txt}")
    print(f"[background] This is fold-invariant and shared identically by Path A, "
          f"Path B, and Path C — no per-fold HVG union is used.")
    return genes


def check_background_consistency(name, genes, background):
    bg_set = set(background)
    missing = [g for g in genes if g not in bg_set]
    frac = len(missing) / len(genes) if genes else 0.0
    is_valid = frac <= MAX_TOLERABLE_MISSING_FRACTION
    return is_valid, missing, frac

## LOCAL hypergeometric enrichment

In [ ]:
def _atomic_to_csv(df, path):
    dirn = os.path.dirname(path)
    with tempfile.NamedTemporaryFile("w", dir=dirn, suffix=".tmp", delete=False) as tmp:
        tmp_path = tmp.name
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)


def get_or_download_library(lib_name, cache_dir=LIBRARY_CACHE_DIR):
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"{lib_name}.json")
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)

    print(f"[library] Downloading '{lib_name}' once (will be cached locally)...")
    lib_dict = gp.get_library(name=lib_name, organism="Human")
    lib_dict = {term: [g.upper() for g in genes] for term, genes in lib_dict.items()}
    with open(cache_path, "w") as f:
        json.dump(lib_dict, f)
    return lib_dict


def local_hypergeometric_enrichment(query_genes, library_dict, background_genes):
    bg_set = set(background_genes)
    query_set = set(query_genes) & bg_set  # enforce query ⊆ background
    N = len(bg_set)          # background population size
    n = len(query_set)       # query sample size

    rows = []
    for term, term_genes in library_dict.items():
        term_in_bg = set(term_genes) & bg_set
        K = len(term_in_bg)  # number of "successes" in population
        if K == 0:
            continue
        overlap = query_set & term_in_bg
        k = len(overlap)     # observed successes in sample
        if k == 0:
            continue
        # P(X >= k) for hypergeometric(N, K, n)
        pval = hypergeom.sf(k - 1, N, K, n)
        rows.append({
            "Term": term,
            "N_Overlap_Genes": k,
            "Term_Size_In_Background": K,
            "Overlap_Genes": ";".join(sorted(overlap)),
            "P-value": pval,
        })

    if not rows:
        return pd.DataFrame(columns=[
            "Term", "N_Overlap_Genes", "Term_Size_In_Background",
            "Overlap_Genes", "P-value", "Adjusted P-value",
        ])

    df = pd.DataFrame(rows)
    # BH-FDR correction WITHIN this library's tested terms (matches standard
    # Enrichr methodology; explicitly documented rather than assumed).
    df["Adjusted P-value"] = multipletests(df["P-value"], method="fdr_bh")[1]
    return df.sort_values("Adjusted P-value").reset_index(drop=True)


def run_enrichment_for_path(name, genes, background, outdir, libraries):
    path_outdir = os.path.join(outdir, name)
    os.makedirs(path_outdir, exist_ok=True)

    all_lib_results = []
    for lib in libraries:
        cache_file = os.path.join(path_outdir, f"{name}_{lib}.csv")
        if os.path.exists(cache_file):
            df = pd.read_csv(cache_file)
        else:
            lib_dict = get_or_download_library(lib)
            df = local_hypergeometric_enrichment(genes, lib_dict, background)
            df["Gene_set"] = lib
            _atomic_to_csv(df, cache_file)
        all_lib_results.append(df)

    res = pd.concat(all_lib_results, ignore_index=True) if all_lib_results else pd.DataFrame()
    if res.empty:
        print(f"[{name}] No terms with any overlap found across {len(libraries)} libraries.")
        return res

    res = res.sort_values("Adjusted P-value").reset_index(drop=True)
    _atomic_to_csv(res, os.path.join(path_outdir, f"{name}_full_results.csv"))

    sig = res[res["Adjusted P-value"] < FDR_CUTOFF]
    _atomic_to_csv(sig, os.path.join(path_outdir, f"{name}_significant_FDR{FDR_CUTOFF}.csv"))

    print(f"[{name}] {len(res)} terms tested across {len(libraries)} libraries, "
          f"{len(sig)} significant at FDR<{FDR_CUTOFF}")
    return res

## Gene-attribution summary

In [ ]:
def summarize_gene_contribution(name, res_df, query_genes):
    sig = res_df[res_df["Adjusted P-value"] < FDR_CUTOFF]
    contributing = set()
    for overlap_str in sig["Overlap_Genes"].dropna():
        contributing |= set(overlap_str.split(";"))
    contributing &= set(query_genes)
    return {
        "Path": name,
        "N_query_genes": len(query_genes),
        "N_significant_terms": len(sig),
        "N_query_genes_contributing_to_any_significant_term": len(contributing),
        "Contributing_genes": ", ".join(sorted(contributing)),
    }

## Plotting

In [ ]:
def plot_top_terms(name, res_df, outdir, libraries):
    path_outdir = os.path.join(outdir, name)
    for lib in libraries:
        sub = res_df[res_df["Gene_set"] == lib].sort_values("Adjusted P-value").head(TOP_N_TERMS_FOR_PLOTS)
        if sub.empty:
            continue
        sub = sub.iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(sub))))
        safe_p = sub["Adjusted P-value"].clip(lower=1e-300)
        ax.barh(sub["Term"], -safe_p.apply(math.log10))
        ax.set_xlabel("-log10(Adjusted P-value)")
        title_suffix = " [NEGATIVE CONTROL]" if name in NEGATIVE_CONTROL_PATHS else ""
        ax.set_title(f"{name}{title_suffix}\n{lib}")
        fig.tight_layout()
        fig.savefig(os.path.join(path_outdir, f"{name}_{lib}_top_terms.png"), dpi=200)
        plt.close(fig)


def plot_significant_counts_summary(all_results, outdir, alias_map, libraries):
    counts = []
    for path_name, canonical in alias_map.items():
        res = all_results.get(canonical)
        if res is None or res.empty:
            continue
        for lib in libraries:
            n_sig = ((res["Gene_set"] == lib) & (res["Adjusted P-value"] < FDR_CUTOFF)).sum()
            counts.append({
                "Path": path_name + (" [NEG. CTRL]" if path_name in NEGATIVE_CONTROL_PATHS else ""),
                "Library": lib,
                "N_significant": n_sig,
            })
    counts_df = pd.DataFrame(counts)
    if counts_df.empty:
        print("[plot_significant_counts_summary] No data to plot.")
        return counts_df

    pivot = counts_df.pivot(index="Path", columns="Library", values="N_significant").fillna(0)
    fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(pivot)), 6))
    pivot.plot(kind="bar", ax=ax)
    ax.set_ylabel(f"# significant terms (FDR < {FDR_CUTOFF}, within-library BH correction)")
    ax.set_title("Enrichment strength by path and library\n"
                  "([NEG. CTRL] = random-gene negative control; excluded paths due to "
                  "background mismatch are omitted, see background_validity_report.csv)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "summary_significant_counts_by_path.png"), dpi=200)
    plt.close(fig)
    counts_df.to_csv(os.path.join(outdir, "summary_significant_counts_by_path.csv"), index=False)
    return counts_df


def build_comparison_table(all_results, outdir, alias_map, libraries, gene_contribution_rows):
    rows = []
    for path_name, canonical in alias_map.items():
        res = all_results.get(canonical)
        if res is None or res.empty:
            continue
        available_libs_here = [lib for lib in PREFERRED_LIBRARY_ORDER if (res["Gene_set"] == lib).any()]
        for lib in available_libs_here:
            sub = res[(res["Gene_set"] == lib) & (res["Adjusted P-value"] < FDR_CUTOFF)]
            sub = sub.sort_values("Adjusted P-value").head(TOP_N_TERMS_FOR_PLOTS)
            for _, r in sub.iterrows():
                rows.append({
                    "Path": path_name,
                    "Shares_geneset_with": canonical if canonical != path_name else "",
                    "Is_negative_control": path_name in NEGATIVE_CONTROL_PATHS,
                    "Library": lib,
                    "FDR_scope": "within-library (BH)",
                    "GO_Term": r["Term"],
                    "Adjusted_P_value": r["Adjusted P-value"],
                    "Overlap_Genes": r["Overlap_Genes"],
                    "N_Overlap_Genes": r["N_Overlap_Genes"],
                })
    comp = pd.DataFrame(rows)
    comp.to_csv(os.path.join(outdir, "cross_path_GO_comparison.csv"), index=False)

    contrib_df = pd.DataFrame(gene_contribution_rows)
    contrib_df.to_csv(os.path.join(outdir, "gene_level_contribution_summary.csv"), index=False)

    return comp

## Export the Shared Background

In [ ]:
BACKGROUND_GENES_FROM_PANEL = export_shared_background_full_panel(
    adata_paths=[
        "/content/drive/.../bio_dataset.h5ad",

    ]
)

## Main Pipeline

In [ ]:
def main():
    os.makedirs(OUTDIR, exist_ok=True)
    background = load_background()

    gene_sets = build_gene_sets_from_folds(FOLD_GENE_SETS, OUTDIR, top_n=AGGREGATION_TOP_N)
    unique_gene_sets, alias_map = deduplicate_gene_sets(gene_sets)

    print()
    all_results = {}
    background_validity_rows = []
    gene_contribution_rows = []
    excluded_paths = []

    for name, raw_genes in unique_gene_sets.items():
        curated_genes, curation_log = curate_query_genes(raw_genes)

        if curation_log["aliases_expanded"]:
            print(f" [{name}] alias-expanded: {curation_log['aliases_expanded']}")
        if curation_log["artifacts_removed"]:
            print(f" [{name}] removed known technical-artifact genes from ENRICHMENT QUERY "
                  f"(not from upstream feature selection): {list(curation_log['artifacts_removed'])}")

        is_valid, missing, frac = check_background_consistency(name, curated_genes, background)
        background_validity_rows.append({
            "Path": name,
            "N_query_genes": len(curated_genes),
            "N_missing_from_background": len(missing),
            "Missing_fraction": round(frac, 3),
            "Missing_genes": ", ".join(missing),
            "Included_in_comparison": is_valid,
        })

        if not is_valid:
            print(f" [{name}] EXCLUDED from cross-path comparison: {len(missing)}/{len(curated_genes)} "
                  f"query genes ({frac:.0%}) are absent from the background universe. The "
                  f"hypergeometric test's denominator would not represent this gene list's "
                  f"actual sampling frame. Resolve by matching this path's preprocessing/HVG "
                  f"definition to the shared background before re-including it.")
            excluded_paths.append(name)
            continue

        res = run_enrichment_for_path(name, curated_genes, background, OUTDIR, GENE_SET_LIBRARIES)
        if res.empty:
            continue

        plot_top_terms(name, res, OUTDIR, GENE_SET_LIBRARIES)
        all_results[name] = res
        gene_contribution_rows.append(summarize_gene_contribution(name, res, curated_genes))

    pd.DataFrame(background_validity_rows).to_csv(
        os.path.join(OUTDIR, "background_validity_report.csv"), index=False
    )

    comparison = build_comparison_table(all_results, OUTDIR, alias_map, GENE_SET_LIBRARIES, gene_contribution_rows)
    print("\n=== Cross-path GO comparison (excluded paths listed in background_validity_report.csv) ===")
    print(comparison.to_string(index=False) if not comparison.empty else
          "No terms passed the FDR cutoff for any valid path.")

    plot_significant_counts_summary(all_results, OUTDIR, alias_map, GENE_SET_LIBRARIES)
    print(f"\nAll results saved under: {os.path.abspath(OUTDIR)}/")

    if excluded_paths:
        print(f"\n[EXCLUDED PATHS — background mismatch]: {excluded_paths}")

    print("\nREMINDER: BH-FDR correction is applied WITHIN each library's tested terms. "
          "Comparisons across paths/libraries are descriptive, not independently "
          "significance-tested (consistent with the n=4 donor limitation noted for the "
          "classifier results). Paths marked [NEG. CTRL] are random-gene negative controls — "
          "'significant' terms there represent the expected false-positive rate under "
          "multiple testing, not real biology. See gene_level_contribution_summary.csv for "
          "how many of each path's 10 query genes actually drove any significant term "
          "(important given the low statistical power of n=10 gene lists).")

    return gene_sets, alias_map, all_results, comparison


if __name__ == "__main__":
    gene_sets, alias_map, all_results, comparison = main()